<a href="https://colab.research.google.com/github/sivashobana547/ClinicalInsight--Automated_Medical_Report_Processor/blob/main/Medical_analyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update
!apt-get install -y tesseract-ocr poppler-utils
!pip install pytesseract pdf2image pillow groq

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.4 MB]
Get:14 http

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_files = list(uploaded.keys())
print("Uploaded patient reports:")
for f in pdf_files:
    print("-", f)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import re
import pytesseract
from pdf2image import convert_from_path
from groq import Groq
from google.colab import files
from datetime import datetime

In [ ]:
from google.colab import userdata

ENABLE_PHI_REIDENTIFICATION = True
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [ ]:
def extract_text_from_pdf(pdf_path):
    images = convert_from_path(pdf_path)
    text = ""
    for img in images:
        text += pytesseract.image_to_string(img) + "\n"
    return text.strip()

In [ ]:
def anonymize_phi_text(text):
    phi_map = {}

    # Patient Name
    name_match = re.search(r"Patient Name:\s*(.*)", text)
    if name_match:
        name = name_match.group(1).strip()
        phi_map["<<PATIENT_NAME>>"] = name
        text = text.replace(name, "<<PATIENT_NAME>>")

    # Phone numbers
    phones = re.findall(r"\b\d{10}\b", text)
    for i, phone in enumerate(phones):
        token = f"<<PHONE_{i}>>"
        phi_map[token] = phone
        text = text.replace(phone, token)

    # Emails
    emails = re.findall(r"\S+@\S+", text)
    for i, email in enumerate(emails):
        token = f"<<EMAIL_{i}>>"
        phi_map[token] = email
        text = text.replace(email, token)

    return text, phi_map

In [ ]:
def safe_json_loads(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError("No valid JSON found in model output")

In [ ]:
EVENT_SYSTEM_PROMPT = """
Return ONLY valid JSON.
Extract:
- report_date (YYYY-MM-DD if available)
- medical_events (date, event, category)
"""

def extract_events(text):
    prompt = f"""
Clinical Text:
{text}

Return JSON:
{{
  "report_date": "",
  "medical_events": [
    {{
      "date": "",
      "event": "",
      "category": ""
    }}
  ]
}}
"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": EVENT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return safe_json_loads(response.choices[0].message.content)

In [ ]:
reports = []
global_phi_map = {}

for pdf in pdf_files:
    raw_text = extract_text_from_pdf(pdf)
    anonymized_text, phi_map = anonymize_phi_text(raw_text)
    global_phi_map.update(phi_map)

    extracted = extract_events(anonymized_text)
    extracted["raw_text"] = anonymized_text
    reports.append(extracted)

In [ ]:
def parse_date(d):
    try:
        return datetime.strptime(d, "%Y-%m-%d")
    except:
        return datetime.min

reports_sorted = sorted(reports, key=lambda x: parse_date(x.get("report_date")))
latest_report = reports_sorted[-1]
previous_reports = reports_sorted[:-1]

In [ ]:
HISTORY_PROMPT = """
Extract historical medical information ONLY.

Return JSON:
{
  "past_diseases": [],
  "past_medications": [],
  "past_consultations": []
}
"""

historical_text = "\n\n".join(r["raw_text"] for r in previous_reports)

if not historical_text.strip():  # Check if historical_text is empty or contains only whitespace
    history_structured = {
        "past_diseases": [],
        "past_medications": [],
        "past_consultations": []
    }
else:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": HISTORY_PROMPT},
            {"role": "user", "content": historical_text}
        ],
        temperature=0
    )

    history_structured = safe_json_loads(response.choices[0].message.content)

In [ ]:
SUMMARY_PROMPT = """
Generate a concise medical history summary for a doctor
based on the structured data below.
Do NOT mention latest report findings.
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "Text only."},
        {
            "role": "user",
            "content": SUMMARY_PROMPT + json.dumps(history_structured, indent=2)
        }
    ],
    temperature=0
)

medical_history_summary = response.choices[0].message.content

In [ ]:
ABNORMALITY_PROMPT = """
You are a clinical decision-support system.

Tasks:
- Identify abnormal lab values
- Assess severity (mild/moderate/severe)
- Provide recommendations
- Suggest general medications (non-prescriptive)
- If severe, advise doctor consultation timeframe

Return JSON only.
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": ABNORMALITY_PROMPT},
        {"role": "user", "content": latest_report["raw_text"]}
    ],
    temperature=0
)

latest_report_analysis = safe_json_loads(
    response.choices[0].message.content
)

In [ ]:
def reidentify_phi(data, phi_map, enable):
    if not enable:
        data["phi_anonymized"] = True
        return data

    data_str = json.dumps(data)
    for token, original in phi_map.items():
        data_str = data_str.replace(token, original)

    restored = json.loads(data_str)
    restored["phi_anonymized"] = False
    return restored

In [ ]:
final_output = {
    "medical_history_summary": medical_history_summary,
    "past_diseases": history_structured["past_diseases"],
    "past_medications": history_structured["past_medications"],
    "past_consultations": history_structured["past_consultations"],
    "latest_report_analysis": latest_report_analysis
}

final_output = reidentify_phi(
    final_output,
    global_phi_map,
    ENABLE_PHI_REIDENTIFICATION
)

print(json.dumps(final_output, indent=2))

In [ ]:
a = input()
b = a[0]
c = a[-1]
print(b<c)
print(b>c)

87
False
True
